# Получение часовых поясов для аэропортов

Для задачи получения часовых поясов для аэропортов, воспользуемся [данным датасетом с Kaggle](https://www.kaggle.com/datasets/samvelkoch/global-airports-iata-icao-timezone-geo). В отличие используемого ранее датасета для получения IATA кодов, который содержал колонку `Timezone` в формате количества часов от формата UTC (например, значения `10` или `-3`) , данный содержит часовые пояса в формате IANA (например, "America/New_York", "Europe/London"), что будет подходящей опицей для библиотеки zoneinfo, с помощью которой можно получить текущее время для конкретного часового пояса.

## Импорт

In [1]:
import pandas as pd
import sqlite3

## Соединение и сохранение

In [2]:
conn = sqlite3.connect('../data/flights_info.db')
df_db = pd.read_sql(con=conn, sql='SELECT * FROM airports_info;')
df_db.head()

,name_en,city_en,country_en,IATA,name_ru,city_ru,country_ru
0,Goroka Airport,Goroka,Papua New Guinea,GKA,Аэропорт Горока,Горока,Папуа - Новая Гвинея
1,Madang Airport,Madang,Papua New Guinea,MAG,Аэропорт Маданга,Маданг,Папуа - Новая Гвинея
2,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,Аэропорт Маунт-Хаген-Кагамуга,Гора Хаген,Папуа - Новая Гвинея
3,Nadzab Airport,Nadzab,Papua New Guinea,LAE,Аэропорт Надзаб,Надзаб,Папуа - Новая Гвинея
4,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,Международный аэропорт Порт-Морсби Джексонс,Порт-Морсби,Папуа - Новая Гвинея


In [3]:
# Выделяем только интересующие колонки - часовой пояс и IATA код. Соединение будет по IATA коду
df_timezones = pd.read_csv('../data/airports.csv')[['IATA', 'TimeZone']].rename({'TimeZone': 'timezone'}, axis=1)
df_timezones.head()

,IATA,timezone
0,NTY,Africa/Johannesburg
1,CVN,America/Denver
2,CVS,America/Denver
3,SCM,America/Nome
4,KPI,Asia/Kuching


In [5]:
df_left_join = df_db.merge(right=df_timezones, how='left', on='IATA')
df_left_join.head()

,name_en,city_en,country_en,IATA,name_ru,city_ru,country_ru,timezone
0,Goroka Airport,Goroka,Papua New Guinea,GKA,Аэропорт Горока,Горока,Папуа - Новая Гвинея,Pacific/Port_Moresby
1,Madang Airport,Madang,Papua New Guinea,MAG,Аэропорт Маданга,Маданг,Папуа - Новая Гвинея,Pacific/Port_Moresby
2,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,Аэропорт Маунт-Хаген-Кагамуга,Гора Хаген,Папуа - Новая Гвинея,Pacific/Port_Moresby
3,Nadzab Airport,Nadzab,Papua New Guinea,LAE,Аэропорт Надзаб,Надзаб,Папуа - Новая Гвинея,Pacific/Port_Moresby
4,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,Международный аэропорт Порт-Морсби Джексонс,Порт-Морсби,Папуа - Новая Гвинея,Pacific/Port_Moresby


In [6]:
df_left_join.isna().sum()

name_en         0
city_en        39
country_en      0
IATA            0
name_ru         0
city_ru        39
country_ru      0
timezone      918
dtype: int64

In [7]:
df_left_join.shape

(6072, 8)

Примерно для каждого шестого аэропорта не удалось определить часовой пояс

In [9]:
df_left_join[df_left_join['timezone'].isna()]

,name_en,city_en,country_en,IATA,name_ru,city_ru,country_ru,timezone
53,Forestville Airport,Forestville,Canada,YFE,Аэропорт Форествилля,Лесной городок,Канада,NaN
201,Blida Airport,Blida,Algeria,QLD,Аэропорт Блида,Блида,Алжир,NaN
202,Bou Saada Airport,Bou Saada,Algeria,BUJ,Аэропорт Бу-Саада,Бу Саада,Алжир,NaN
209,Mecheria Airport,Mecheria,Algeria,MZW,Аэропорт Мехерия,Мечерия,Алжир,NaN
216,Ech Cheliff Airport,Ech-cheliff,Algeria,CFK,Аэропорт Эч-Челифф,Эхо-челифф,Алжир,NaN
...,...,...,...,...,...,...,...,...
6064,Xingcheng Air Base,NaN,China,XEN,Военно-воздушная база Синчэн,NaN,Китай,NaN
6065,Lefkoniko Airport,Geçitkale,Cyprus,GEC,Аэропорт Лефконико,Гечиткале,Кипр,NaN
6066,Songwe Airport,Mbeya,Tanzania,MBI,Аэропорт Сонгве,Мбея,Танзания,NaN
6067,Bilogai-Sugapa Airport,Sugapa-Papua Island,Indonesia,UGU,Аэропорт Билогай-Сугапа,Сугапа-остров Папуа,Индонезия,NaN


In [10]:
df_left_join.to_sql('airports_info', conn, if_exists='replace', index=False)

6072